<a href="https://colab.research.google.com/github/Aseg1999/INTELIGENCIA-ARTIFICAL-parte-2/blob/main/a_estrella_videojuego.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import heapq
import time
from copy import deepcopy
from IPython.display import clear_output

# ==========================================
# MAPA DEL VIDEOJUEGO
# P = jugador
# M = meta
# # = obstáculo
# . = camino libre
# ==========================================
mapa_original = [
    ['P', '.', '.', '#', '.', '.', '.'],
    ['#', '#', '.', '#', '.', '#', '.'],
    ['.', '.', '.', '.', '.', '#', '.'],
    ['.', '#', '#', '#', '.', '.', '.'],
    ['.', '.', '.', '.', '#', '.', '.'],
    ['#', '.', '#', '.', '.', '.', 'M']
]

FILAS = len(mapa_original)
COLUMNAS = len(mapa_original[0])

# ==========================================
# BUSCAR POSICIÓN INICIAL Y META
# ==========================================
def buscar_posiciones(mapa):
    inicio = None
    meta = None

    for i in range(FILAS):
        for j in range(COLUMNAS):
            if mapa[i][j] == 'P':
                inicio = (i, j)
            elif mapa[i][j] == 'M':
                meta = (i, j)

    return inicio, meta

# ==========================================
# HEURÍSTICA MANHATTAN
# ==========================================
def heuristica(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

# ==========================================
# IMPRIMIR MAPA
# ==========================================
def imprimir_mapa(mapa):
    for fila in mapa:
        print(" ".join(fila))
    print()

# ==========================================
# RECONSTRUIR CAMINO FINAL
# ==========================================
def reconstruir_camino(vino_de, inicio, meta):
    actual = meta
    camino = []

    while actual != inicio:
        camino.append(actual)
        actual = vino_de[actual]

    camino.append(inicio)
    camino.reverse()
    return camino

# ==========================================
# MOSTRAR MAPA CON VISITADOS Y ACTUAL
# ==========================================
def crear_mapa_visual(mapa, visitados, actual, frontera):
    mapa_temp = deepcopy(mapa)

    for v in visitados:
        if mapa_temp[v[0]][v[1]] == '.':
            mapa_temp[v[0]][v[1]] = '·'   # visitados

    for f in frontera:
        x, y = f
        if mapa_temp[x][y] == '.':
            mapa_temp[x][y] = '?'         # frontera o pendientes

    if actual and mapa_temp[actual[0]][actual[1]] not in ('P', 'M'):
        mapa_temp[actual[0]][actual[1]] = 'X'  # nodo actual

    return mapa_temp

# ==========================================
# ALGORITMO A* MÁS ILUSTRADO
# ==========================================
def a_estrella_ilustrado(mapa, pausa=2):
    inicio, meta = buscar_posiciones(mapa)

    movimientos = [
        (0, 1),   # derecha
        (1, 0),   # abajo
        (0, -1),  # izquierda
        (-1, 0)   # arriba
    ]

    cola_prioridad = []
    heapq.heappush(cola_prioridad, (0, inicio))

    vino_de = {inicio: None}
    costo_g = {inicio: 0}
    visitados = set()

    paso = 0

    print("MAPA INICIAL")
    imprimir_mapa(mapa)
    time.sleep(3)

    while cola_prioridad:
        paso += 1

        # sacar nodo con menor prioridad
        prioridad_actual, actual = heapq.heappop(cola_prioridad)

        if actual in visitados:
            continue

        visitados.add(actual)

        g_actual = costo_g[actual]
        h_actual = heuristica(actual, meta)
        f_actual = g_actual + h_actual

        frontera_actual = [nodo for _, nodo in cola_prioridad]

        clear_output(wait=True)
        print(f"PASO {paso}")
        print(f"Nodo actual: {actual}")
        print(f"g(n) = {g_actual}  -> costo real desde el inicio")
        print(f"h(n) = {h_actual}  -> estimación hasta la meta")
        print(f"f(n) = {f_actual}  -> suma de g + h")
        print()

        mapa_visual = crear_mapa_visual(mapa, visitados, actual, frontera_actual)
        imprimir_mapa(mapa_visual)

        print("Leyenda:")
        print("P = inicio")
        print("M = meta")
        print("# = obstáculo")
        print(". = camino libre")
        print("· = ya explorado")
        print("? = pendiente por explorar")
        print("X = nodo actual")
        print()

        if actual == meta:
            print("¡La meta fue encontrada!")
            time.sleep(3)

            camino = reconstruir_camino(vino_de, inicio, meta)

            for paso_camino in camino:
                if mapa[paso_camino[0]][paso_camino[1]] not in ('P', 'M'):
                    mapa[paso_camino[0]][paso_camino[1]] = '*'

            clear_output(wait=True)
            print("CAMINO FINAL ENCONTRADO")
            imprimir_mapa(mapa)

            print("Ruta final:")
            print(camino)
            print(f"Cantidad de pasos: {len(camino) - 1}")
            return camino

        print("Revisando vecinos...\n")
        time.sleep(pausa)

        for mov in movimientos:
            vecino = (actual[0] + mov[0], actual[1] + mov[1])

            print(f"Vecino analizado: {vecino}")

            # verificar si está dentro del mapa
            if not (0 <= vecino[0] < FILAS and 0 <= vecino[1] < COLUMNAS):
                print("-> Está fuera del mapa, se descarta.\n")
                time.sleep(1.5)
                continue

            # verificar si es obstáculo
            if mapa[vecino[0]][vecino[1]] == '#':
                print("-> Es un obstáculo (#), se descarta.\n")
                time.sleep(1.5)
                continue

            nuevo_costo = costo_g[actual] + 1

            if vecino not in costo_g or nuevo_costo < costo_g[vecino]:
                costo_g[vecino] = nuevo_costo
                h_vecino = heuristica(vecino, meta)
                f_vecino = nuevo_costo + h_vecino

                heapq.heappush(cola_prioridad, (f_vecino, vecino))
                vino_de[vecino] = actual

                print(f"-> Se agrega a la cola.")
                print(f"   g = {nuevo_costo}")
                print(f"   h = {h_vecino}")
                print(f"   f = {f_vecino}\n")
            else:
                print("-> Ya tenía un camino mejor, no se actualiza.\n")

            time.sleep(1.5)

    print("No se encontró camino.")
    return None

# ==========================================
# EJECUCIÓN
# ==========================================
a_estrella_ilustrado(deepcopy(mapa_original), pausa=2)